# Notebook 02: Curbe Pfa și Pd — caracteristicile de performanță

Acest notebook generează curbele $P_{fa}(T)$ și $P_d(\text{SNR})$ pentru detectoarele SL-GLRT și ML-GLRT.

Aceste curbe sunt cerute explicit în Capitolul IV al lucrării (secțiunea 4.5):
- **Fig. 4.5**: $P_{fa}$ vs prag $T$ (corespunde cu Fig. 1 din Pauciullo et al. 2018)
- **Fig. 4.6**: $P_d$ vs SNR pentru $P_{fa}$ fixată (corespunde cu Fig. 3 din Pauciullo et al. 2018)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from glrt_sar.detectors import SystemGeometry, SLGLRT, MLGLRT
from glrt_sar.simulation import compute_pfa_curve, compute_pd_curve, threshold_for_pfa
from glrt_sar.config import create_sentinel1_geometry
from glrt_sar.utils import plot_pfa_curve, plot_pd_curve

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Setup geometrie

In [ ]:
# Două configurații de stivă: N=30 și N=50
def make_geom(N):
    rng = np.random.default_rng(42)
    bperp = rng.uniform(-150, 150, N)
    bperp[0] = 0
    btemp_days = np.arange(N) * 6
    return create_sentinel1_geometry(bperp, btemp_days, slant_range=800e3)

geom_30 = make_geom(30)
geom_50 = make_geom(50)
print(f'Geom N=30: Rayleigh elev={geom_30.rayleigh_elevation:.1f}m, vel={geom_30.rayleigh_velocity*100:.2f}cm/an')
print(f'Geom N=50: Rayleigh elev={geom_50.rayleigh_elevation:.1f}m, vel={geom_50.rayleigh_velocity*100:.2f}cm/an')

## 2. Curbe Pfa vs prag T

Generăm curbele pentru SL-GLRT și ML-GLRT cu L=9 (fereastră 3×3) și L=25 (5×5).

**ATENȚIE**: simulările Monte Carlo cu $P_{fa}$ mic (e.g. $10^{-4}$) necesită multe realizări (>10^5). Notebook-ul folosește n_samples=50_000, care este suficient pentru $P_{fa} \geq 10^{-3}$. Pentru valori mai mici, mărește n_samples.

In [ ]:
thresholds = np.linspace(0.05, 0.6, 40)
n_samples_pfa = 30_000  # crește pentru curbe mai netede

pfa_curves = {}

# SL-GLRT (N=30 și N=50)
for geom, label_N in [(geom_30, 30), (geom_50, 50)]:
    det_sl = SLGLRT(geom)
    print(f'SL-GLRT, N={label_N}...')
    pfa_curves[f'SL-GLRT N={label_N}'] = compute_pfa_curve(
        det_sl, thresholds, n_samples=n_samples_pfa, L=1, verbose=False
    )

# ML-GLRT cu L=9, N=30 și N=50
for geom, label_N in [(geom_30, 30), (geom_50, 50)]:
    det_ml = MLGLRT(geom, window_size=3)
    print(f'ML-GLRT L=9, N={label_N}...')
    pfa_curves[f'ML-GLRT L=9 N={label_N}'] = compute_pfa_curve(
        det_ml, thresholds, n_samples=n_samples_pfa, L=9, verbose=False
    )

print('Gata!')

In [ ]:
# Plot Pfa vs T - similar Fig. 1 din Pauciullo 2018 / Fig. 2 din De Maio 2009
fig, ax = plt.subplots(figsize=(10, 7))

styles = {
    'SL-GLRT N=30': ('-', 'C0', 2),
    'SL-GLRT N=50': ('--', 'C0', 2),
    'ML-GLRT L=9 N=30': ('-', 'C3', 2),
    'ML-GLRT L=9 N=50': ('--', 'C3', 2),
}

for name, pfa in pfa_curves.items():
    ls, color, lw = styles[name]
    ax.semilogy(thresholds, np.maximum(pfa, 1e-6), ls,
                color=color, linewidth=lw, label=name)

ax.set_xlabel(r'Pragul $T$', fontsize=12)
ax.set_ylabel(r'Probabilitatea de alarmă falsă $P_{fa}$', fontsize=12)
ax.set_title('Fig. 4.5. $P_{fa}$ vs $T$ pentru SL-GLRT și ML-GLRT (L=9)', fontsize=13)
ax.legend(fontsize=11, loc='lower left')
ax.grid(True, which='both', alpha=0.3)
ax.set_ylim(1e-5, 1)
plt.tight_layout()
plt.savefig('fig_4_5_pfa_curves.png', dpi=150, bbox_inches='tight')
plt.show()

**Observații Fig. 4.5:**

- Curbele pentru SL-GLRT cu N=30 vs N=50 sunt deplasate: pentru aceeași $P_{fa}$, N mai mare permite prag T mai mic (CFAR funcționează, dar pragul depinde de N).
- Curbele ML-GLRT L=9 sunt deplasate la stânga față de SL-GLRT (la aceeași $P_{fa}$, pragul T este mai mic pentru ML). Aceasta este consecința faptului că ML-GLRT mediază peste 9 lookuri, statistic stabilizându-se.

## 3. Curbe Pd vs SNR (la $P_{fa} = 10^{-4}$ fixat)

Pentru fiecare detector, calibrăm pragul T pentru a obține $P_{fa} = 10^{-4}$, apoi măsurăm $P_d$ în funcție de SNR.

In [ ]:
target_pfa = 1e-4
n_samples_threshold = 200_000  # pentru calibrare prag (trebuie > 10/target_pfa)
n_samples_pd = 3_000
snr_range = np.arange(-15, 21, 2)  # dB

# Calibrare praguri
print('Calibrare praguri pentru Pfa = 1e-4...')
geom = geom_30  # folosim N=30 pentru această analiză

det_sl = SLGLRT(geom)
det_ml9 = MLGLRT(geom, window_size=3)
det_ml25 = MLGLRT(geom, window_size=5)

T_sl = threshold_for_pfa(det_sl, target_pfa, n_samples=n_samples_threshold, L=1)
T_ml9 = threshold_for_pfa(det_ml9, target_pfa, n_samples=n_samples_threshold, L=9)
T_ml25 = threshold_for_pfa(det_ml25, target_pfa, n_samples=n_samples_threshold, L=25)

print(f'  T_SL    = {T_sl:.4f}')
print(f'  T_ML9   = {T_ml9:.4f}')
print(f'  T_ML25  = {T_ml25:.4f}')

In [ ]:
# Calcul Pd vs SNR
print('Calcul Pd vs SNR pentru SL-GLRT...')
pd_sl = compute_pd_curve(det_sl, T_sl, snr_range, n_samples=n_samples_pd, L=1, verbose=False)

print('Calcul Pd vs SNR pentru ML-GLRT L=9...')
pd_ml9 = compute_pd_curve(det_ml9, T_ml9, snr_range, n_samples=n_samples_pd, L=9, verbose=False)

print('Calcul Pd vs SNR pentru ML-GLRT L=25...')
pd_ml25 = compute_pd_curve(det_ml25, T_ml25, snr_range, n_samples=n_samples_pd, L=25, verbose=False)

print('Gata!')

In [ ]:
# Plot Pd vs SNR
fig, ax = plt.subplots(figsize=(10, 7))

ax.plot(snr_range, pd_sl, 'o-', color='C0', linewidth=2, markersize=6,
        label='SL-GLRT (L=1)')
ax.plot(snr_range, pd_ml9, 's-', color='C3', linewidth=2, markersize=6,
        label='ML-GLRT (L=9, fereastră 3×3)')
ax.plot(snr_range, pd_ml25, '^-', color='C2', linewidth=2, markersize=6,
        label='ML-GLRT (L=25, fereastră 5×5)')

ax.axhline(0.9, color='gray', linestyle=':', alpha=0.7, label=r'$P_d = 0.9$')

ax.set_xlabel('SNR (dB)', fontsize=12)
ax.set_ylabel(r'Probabilitatea de detecție $P_d$', fontsize=12)
ax.set_title(f'Fig. 4.6. Pd vs SNR la $P_{{fa}}={target_pfa:.0e}$, N={geom.n_images}', fontsize=13)
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.02, 1.02)
plt.tight_layout()
plt.savefig('fig_4_6_pd_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Tabel cu valorile SNR pentru Pd = 0.9 (câștigul de SNR al ML vs SL)
from scipy.interpolate import interp1d

def snr_at_pd(snr_arr, pd_arr, pd_target=0.9):
    """Găsește SNR la care Pd atinge pd_target prin interpolare liniară."""
    # Filtrăm punctele crescătoare
    if pd_arr.max() < pd_target:
        return np.nan
    f = interp1d(pd_arr, snr_arr, bounds_error=False)
    return float(f(pd_target))

snr_sl_09 = snr_at_pd(snr_range, pd_sl)
snr_ml9_09 = snr_at_pd(snr_range, pd_ml9)
snr_ml25_09 = snr_at_pd(snr_range, pd_ml25)

print(f'SNR necesar pentru P_d = 0.9 (P_fa = 1e-4, N = {geom.n_images}):')
print(f'  SL-GLRT:           {snr_sl_09:.2f} dB')
print(f'  ML-GLRT L=9:       {snr_ml9_09:.2f} dB  (câștig: {snr_sl_09 - snr_ml9_09:.2f} dB)')
print(f'  ML-GLRT L=25:      {snr_ml25_09:.2f} dB  (câștig: {snr_sl_09 - snr_ml25_09:.2f} dB)')

## 4. Concluzii

Curbele de mai sus confirmă comportamentul așteptat:

1. **Câștigul de detecție ML-GLRT vs SL-GLRT** crește cu numărul de lookuri L. Pentru $P_d = 0.9$ la $P_{fa} = 10^{-4}$, ML-GLRT cu L=25 obține un câștig SNR de câțiva dB față de SL-GLRT.

2. **Compromisul rezoluție-sensibilitate**: ML-GLRT îmbunătățește sensibilitatea dispersorilor slabi (SNR mic), dar reduce rezoluția spațială (factor L în număr de pixeli analizați).

3. **Costul calibrării**: pentru ML-GLRT, pragul T depinde de L, deci trebuie recalibrat pentru fiecare valoare. Aceasta complică implementarea multilookului adaptiv.

În notebook-ul **03** vom analiza explicit influența parametrilor N și L; în **04** vom aplica detectoarele pe date Sentinel-1 reale.